# NeurIPS Diffusion Evaluation

Train diffusion model, compute FID (AE or GyroSwin latent space), evaluate warm restarts, test FID-vs-convergence.

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append("..")
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [2]:
import omegaconf, yaml
from collections import defaultdict

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

from neugk.diffusion import get_diffusion_runner

from neurips_diff_eval import (
    flux_uq_for_valset, plot_flux_confidence,
    compute_statistics, compute_fid,
    compute_fid_on_latents, extract_gyroswin_latents,
    to_model_space, from_model_space,
    run_trajectory_pair, time_to_convergence,
    plot_correlation_grid, load_reference_flux
)

## 0. Configuration

In [3]:
DATA_PATH = "/local00/bioinf/galletti/preprocessed_kvikio"
AE_CHECKPOINT = "/restricteddata/ukaea/checkpoints/autoencoders/neurips26/simsiam_normal/20260328_172956_442/best.pth"
GYROSWIN_CHECKPOINT = "/system/user/publicwork/galletti/checkpoints/gyroswin_large/pytorch_model.bin"
GKW_RAW_DIR = "/restricteddata/ukaea/gyrokinetics/raw"

ID_VAL = ["iteration_262.h5", "iteration_135.h5", "iteration_8.h5",
          "iteration_232.h5", "iteration_148.h5", "iteration_115.h5"]
OOD_VAL = [f"ood_iteration_{i}.h5" for i in range(5)]
TRAIN_TRAJS = "iteration_{0-5,7-12,14-31,33-82,84-99}.h5"

N_EPOCHS = 100
BATCH_SIZE = 256
LR = 2e-3
VAL_EVERY = 30
N_DENOISING_STEPS = 10

MINIBATCH_OT = True
NOISE_DISTRIBUTION = "gaussian"  # "gaussian" or "mixture"
CONTINUOUS_TIME = True

FID_MODE = "gyroswin"  # "ae" or "gyroswin"
FID_N_COMPONENTS = 512
FID_MAX_SAMPLES = 128
GEN_BATCH_SIZE = 32

## 1. Train Diffusion Model

In [ ]:
cfg = omegaconf.OmegaConf.load(os.path.join(os.path.dirname(AE_CHECKPOINT), "config.yaml"))

with open("dit_config.yaml", "r") as f:
    diff_cfg = omegaconf.DictConfig(yaml.safe_load(f))
cfg.model = diff_cfg.model
cfg.ddp = diff_cfg.ddp
cfg.workflow = "diffusion"

cfg.output_path = "/tmp/diffusion_eval_pipeline"
cfg.dataset.path = DATA_PATH
cfg.dataset.gds_override = False
cfg.dataset.backend = "gds"
cfg.ae_checkpoint = os.path.dirname(AE_CHECKPOINT)
cfg.dataset.training_trajectories = TRAIN_TRAJS
cfg.dataset.validation_trajectories = ID_VAL
cfg.model.latent_dim = 512
cfg.training.batch_size = BATCH_SIZE
cfg.training.learning_rate = LR
cfg.training.n_epochs = N_EPOCHS
cfg.validation.validate_every_n_epochs = VAL_EVERY
cfg.validation.probe = {"targets": []}
cfg.logging.writer = None
cfg.logging.tqdm = True
cfg.training.num_workers = 0
cfg.training.pin_memory = False
cfg.model.diffusion.formulation = "edm"
cfg.model.diffusion.minibatch_ot = MINIBATCH_OT
cfg.model.diffusion.noise_distribution = NOISE_DISTRIBUTION
cfg.model.diffusion.continuous_time = CONTINUOUS_TIME
cfg.dataset.normalization = {
    "df": {"type": "zscore", "agg_axes": [0, 1, 3, 4, 5]},
    "flux": {"type": "zscore", "agg_axes": None}, 
    "phi": {"type": "zscore", "agg_axes": [0, 1, 2]}
}
cfg.ddp.enable = False
cfg.deepspeed.enable = False

runner = get_diffusion_runner(rank=0, cfg=cfg, world_size=1)
print(f"Model: {sum(p.numel() for p in runner.model.parameters())/1e6:.1f}M params")
print(f"Train: {len(runner.trainset)}, Val: {sum(len(v) for v in runner.valsets)}")

Diffusion latent dataset mode: AE
Loaded AE config for normalization from /restricteddata/ukaea/checkpoints/autoencoders/neurips26/simsiam_normal/20260328_172956_442/config.yaml
Loading ['df', 'phi', 'flux'] in dataset


In [ ]:
raw = runner.trainset.__getitem__(100, get_normalized=False, override_latens=True)       
print(f"raw std: {raw.df.std():.4f}, raw mean: {raw.df.mean():.4f}")                     
scale, shift = runner.trainset._get_scale_shift(0, "df", raw.df)                         
print(f"shift shape: {shift.shape}, scale shape: {scale.shape}")                         
print(f"shift: {shift.squeeze()}")                                                       
print(f"scale: {scale.squeeze()}")                                                       
manual = (raw.df - shift) / scale                                                      
print(f"manual normalized std: {manual.std():.4f}") 

In [ ]:
from neugk.plot_utils import plot_nd

sample = runner.trainset.__getitem__(100, get_normalized=True, override_latens=True)
df_in = sample.df.unsqueeze(0).to(runner.device)
cond = sample.conditioning.unsqueeze(0).to(runner.device)

ae = runner.autoencoder
ae.eval()
with torch.no_grad():
    recon = ae(df_in, condition=cond)["df"]

df_gt = df_in[0].cpu()
df_rec = recon[0].cpu()

print(f"Input shape: {df_gt.shape}, Recon shape: {df_rec.shape}")
print(f"AE recon MSE: {(df_gt - df_rec).pow(2).mean():.6f}")
print(f"AE recon rel err: {(df_gt - df_rec).norm() / df_gt.norm():.4f}")

_ = plot_nd(df_gt, df_rec, to_wandb=False)

In [ ]:
logs = runner(skip_eval=True)

In [ ]:
for k, v in logs[-1].items():
    if "info/" in str(k):
        print(f"  {k}: {v:.1f}")

In [ ]:
sample = runner.trainset.__getitem__(0, get_normalized=True, override_latens=True)
cond = sample.conditioning.unsqueeze(0).to(runner.device)

runner.model.eval()
with torch.no_grad():
    gen_out = runner.sample(cond, latent_only=False, steps=N_DENOISING_STEPS)

df_gt = sample.df.cpu()
df_gen = gen_out["df"][0].cpu()

print(f"GT shape: {df_gt.shape}, Gen shape: {df_gen.shape}")
print(f"Gen vs GT rel err: {(df_gt - df_gen).norm() / df_gt.norm():.4f}")

_ = plot_nd(df_gt, df_gen, to_wandb=False)

In [ ]:
epochs = [l["epoch"] for l in logs]
train_loss = [l.get("train/loss", l.get("train/df", np.nan)) for l in logs]
val_epochs = [l["epoch"] for l in logs if "val_traj/avg_flux_rmse" in l]
val_rmse = [l["val_traj/avg_flux_rmse"] for l in logs if "val_traj/avg_flux_rmse" in l]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, train_loss, lw=1)
ax1.set(xlabel="Epoch", ylabel="Loss", title="Training loss")
ax1.grid(True, alpha=0.15)
if val_rmse:
    ax2.plot(val_epochs, val_rmse, "o-", markersize=4)
ax2.set(xlabel="Epoch", ylabel="Avg flux RMSE", title="Validation")
ax2.grid(True, alpha=0.15)
fig.tight_layout()

for l in reversed(logs):
    if "val_plots" in l:
        for name, obj in l["val_plots"].items():
            if hasattr(obj, "savefig"): display(obj)
        break

## 1.5 Flux UQ (ID)

In [ ]:
id_ids, id_means, id_stds, id_gts = flux_uq_for_valset(
    runner, runner.valsets[0], n_samples_per_traj=32, steps=N_DENOISING_STEPS)
plot_flux_confidence(id_ids, id_means, id_stds, id_gts, title="Flux UQ - ID")
print(f"ID flux RMSE: {np.sqrt(((id_means - id_gts)**2).mean()):.4f}")

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

_cond_keys_probe = sorted(runner.cfg.model.conditioning)
train_lats, train_flux, train_cond = [], [], []
for (fi, ti), s in runner.trainset.precomputed_latents.items():
    train_lats.append(np.array(s["x"]).reshape(-1))
    train_flux.append(float(np.squeeze(s["flux"])))
    train_cond.append(np.array([float(np.squeeze(s[k])) for k in _cond_keys_probe]))
X_train = np.stack(train_lats)
y_train_flux = np.array(train_flux)
C_train = np.stack(train_cond)
print(f"training set: {X_train.shape[0]} samples, latent dim={X_train.shape[1]}")

gen_lats = []
runner.model.eval()
for i in tqdm(range(0, len(X_train), GEN_BATCH_SIZE), desc="generating full training set"):
    c = torch.tensor(C_train[i:i+GEN_BATCH_SIZE], dtype=torch.float32, device=runner.device)
    with torch.no_grad():
        z = runner.sample(c, latent_only=True, steps=N_DENOISING_STEPS)
        gen_lats.append(z.cpu().numpy().reshape(z.shape[0], -1))
X_gen_probe = np.concatenate(gen_lats)

PROBE_N_COMPONENTS = 256
PROBE_ALPHA = 1.0

pca_probe = PCA(n_components=PROBE_N_COMPONENTS).fit(X_train)
X_train_pca = pca_probe.transform(X_train)
X_gen_pca = pca_probe.transform(X_gen_probe)
print(f"PCA: {X_train.shape[1]} -> {PROBE_N_COMPONENTS} ({pca_probe.explained_variance_ratio_.sum():.1%} var)")

probe_flux = Ridge(alpha=PROBE_ALPHA).fit(X_train_pca, y_train_flux)
pred_ae_flux = probe_flux.predict(X_train_pca)
pred_gen_flux = probe_flux.predict(X_gen_pca)
rmse_ae = np.sqrt(mean_squared_error(y_train_flux, pred_ae_flux))
rmse_gen = np.sqrt(mean_squared_error(y_train_flux, pred_gen_flux))
print(f"flux probe — RMSE ae: {rmse_ae:.4f}, RMSE gen: {rmse_gen:.4f}")

probe_cond = Ridge(alpha=PROBE_ALPHA).fit(X_train_pca, C_train)
pred_cond_ae = probe_cond.predict(X_train_pca)
pred_cond_gen = probe_cond.predict(X_gen_pca)
for j, k in enumerate(_cond_keys_probe):
    rmse_ae_k = np.sqrt(mean_squared_error(C_train[:, j], pred_cond_ae[:, j]))
    rmse_gen_k = np.sqrt(mean_squared_error(C_train[:, j], pred_cond_gen[:, j]))
    print(f"  {k} — RMSE ae: {rmse_ae_k:.4f}, RMSE gen: {rmse_gen_k:.4f}")

n_cond = len(_cond_keys_probe)
ncols = max(3, (n_cond + 2 + 1) // 2)
fig, axes = plt.subplots(2, ncols, figsize=(5*ncols, 8))
axes = axes.flatten()

ax = axes[0]
ax.scatter(y_train_flux, pred_ae_flux, s=8, alpha=0.3, label=f"ae RMSE={rmse_ae:.3f}")
ax.scatter(y_train_flux, pred_gen_flux, s=12, alpha=0.5, marker="x", label=f"gen RMSE={rmse_gen:.3f}")
lim = [y_train_flux.min(), y_train_flux.max()]
ax.plot(lim, lim, "k--", lw=0.8); ax.set(xlabel="GT flux", ylabel="predicted", title="flux probe")
ax.legend(fontsize=8); ax.grid(True, alpha=0.15)

for j, k in enumerate(_cond_keys_probe):
    ax = axes[1+j]
    rmse_ae_k = np.sqrt(mean_squared_error(C_train[:, j], pred_cond_ae[:, j]))
    rmse_gen_k = np.sqrt(mean_squared_error(C_train[:, j], pred_cond_gen[:, j]))
    ax.scatter(C_train[:, j], pred_cond_ae[:, j], s=8, alpha=0.3, label=f"ae RMSE={rmse_ae_k:.3f}")
    ax.scatter(C_train[:, j], pred_cond_gen[:, j], s=12, alpha=0.5, marker="x", label=f"gen RMSE={rmse_gen_k:.3f}")
    lim = [C_train[:, j].min(), C_train[:, j].max()]
    ax.plot(lim, lim, "k--", lw=0.8); ax.set(xlabel=f"GT {k}", ylabel="predicted", title=f"{k} probe")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.15)

pca2d = PCA(n_components=2).fit(X_train)
z_real = pca2d.transform(X_train)
z_gen = pca2d.transform(X_gen_probe)
ax = axes[1 + n_cond]
ax.scatter(z_real[:, 0], z_real[:, 1], s=8, alpha=0.3, label="ae")
ax.scatter(z_gen[:, 0], z_gen[:, 1], s=12, alpha=0.4, marker="x", label="gen")
ax.set(xlabel="PC1", ylabel="PC2", title="latent PCA"); ax.legend(fontsize=8); ax.grid(True, alpha=0.15)

for i in range(2 + n_cond, len(axes)):
    axes[i].set_visible(False)
fig.tight_layout()

_cond_meta_map_val = {"itg": "ion_temp_grad", "dg": "density_grad"}
valset = runner.valsets[0]
flat_idx_map_val = {(f, t): idx for idx, (f, t) in valset.flat_index_to_file_and_tstep.items()}

val_dfs, val_conds_t, val_cond_all, val_flux_gt, val_fi_list = [], [], [], [], []
for fi in range(len(valset.files)):
    meta = valset.metadata[fi]
    gt_flux = float(np.mean(meta["flux"][-80:]))
    cond_vals = [float(np.squeeze(meta[_cond_meta_map_val.get(k, k)])) for k in _cond_keys_probe]
    for t_idx in range(valset.file_num_samples[fi]):
        if (fi, t_idx) not in flat_idx_map_val:
            continue
        sample = valset[flat_idx_map_val[(fi, t_idx)]]
        if sample.df is None:
            continue
        val_dfs.append(sample.df)
        val_conds_t.append(sample.conditioning if sample.conditioning is not None else torch.zeros(len(_cond_keys_probe)))
        val_cond_all.append(cond_vals)
        val_flux_gt.append(gt_flux)
        val_fi_list.append(fi)

val_lats_real = []
runner.autoencoder.eval()
for i in tqdm(range(0, len(val_dfs), GEN_BATCH_SIZE), desc="encoding val through AE"):
    batch_df = torch.stack(val_dfs[i:i+GEN_BATCH_SIZE]).to(runner.device)
    batch_cond = torch.stack(val_conds_t[i:i+GEN_BATCH_SIZE]).to(runner.device)
    with torch.no_grad():
        z, _ = runner.autoencoder.encode(batch_df, condition=batch_cond)
    val_lats_real.append(z.cpu().numpy().reshape(z.shape[0], -1))

X_val_real = np.concatenate(val_lats_real)
C_val = np.array(val_cond_all)
y_val_gt = np.array(val_flux_gt)

gen_val_lats = []
runner.model.eval()
for i in tqdm(range(0, len(C_val), GEN_BATCH_SIZE), desc="generating val latents"):
    c = torch.tensor(C_val[i:i+GEN_BATCH_SIZE], dtype=torch.float32, device=runner.device)
    with torch.no_grad():
        z = runner.sample(c, latent_only=True, steps=N_DENOISING_STEPS)
        gen_val_lats.append(z.cpu().numpy().reshape(z.shape[0], -1))
X_val_gen = np.concatenate(gen_val_lats)

X_val_real_pca = pca_probe.transform(X_val_real)
pred_val_ae = probe_flux.predict(X_val_real_pca)
X_val_gen_pca = pca_probe.transform(X_val_gen)
pred_val_gen = probe_flux.predict(X_val_gen_pca)
rmse_val_ae = np.sqrt(mean_squared_error(y_val_gt, pred_val_ae))
rmse_val_gen = np.sqrt(mean_squared_error(y_val_gt, pred_val_gen))
print(f"flux probe (val) — RMSE ae: {rmse_val_ae:.4f}, RMSE gen: {rmse_val_gen:.4f}")

import re as _re
fi_labels = {}
for fi in sorted(set(val_fi_list)):
    m = _re.search(r"iteration_(\d+)", valset.files[fi])
    fi_labels[fi] = f"iter_{m.group(1)}" if m else f"f{fi}"

per_file = {}
for i, fi in enumerate(val_fi_list):
    lbl = fi_labels[fi]
    if lbl not in per_file:
        per_file[lbl] = {"ae": [], "gen": [], "gt": y_val_gt[i]}
    per_file[lbl]["ae"].append(pred_val_ae[i])
    per_file[lbl]["gen"].append(pred_val_gen[i])

labels = sorted(per_file.keys())
fig2, ax2 = plt.subplots(figsize=(max(10, len(labels)*1.2), 5))
x_pos = np.arange(len(labels))
w = 0.25
for j, lbl in enumerate(labels):
    d = per_file[lbl]
    ax2.errorbar(j - w/2, np.mean(d["ae"]), yerr=np.std(d["ae"]), fmt="o", capsize=5,
                 color="#1f77b4", mfc="white", mew=1.5, alpha=0.8)
    ax2.errorbar(j + w/2, np.mean(d["gen"]), yerr=np.std(d["gen"]), fmt="s", capsize=5,
                 color="#ff7f0e", mfc="white", mew=1.5, alpha=0.8)
    ax2.scatter(j, d["gt"], marker="x", s=80, color="#d62728", zorder=3)
ax2.errorbar([], [], fmt="o", color="#1f77b4", mfc="white", mew=1.5, label=f"ae RMSE={rmse_val_ae:.3f}")
ax2.errorbar([], [], fmt="s", color="#ff7f0e", mfc="white", mew=1.5, label=f"gen RMSE={rmse_val_gen:.3f}")
ax2.scatter([], [], marker="x", s=80, color="#d62728", label="gt flux")
ax2.set_xticks(x_pos)
ax2.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax2.set(xlabel="trajectory", ylabel="flux", title="flux probe on validation")
ax2.legend(fontsize=8); ax2.grid(True, axis="y", alpha=0.3, ls="--")
fig2.tight_layout()

## 2. FID Evaluation

- **`ae`**: FID in AE bottleneck (fast, no decode)
- **`gyroswin`**: FID using frozen GyroSwin encoder (Inception-FID analog)

In [ ]:
if FID_MODE == "gyroswin":
    sd = torch.load(GYROSWIN_CHECKPOINT, map_location="cpu", weights_only=False)

    with open("../configs/model/multi.yaml") as f:
        gs_cfg = omegaconf.DictConfig(yaml.safe_load(f))

    from neugk.gyroswin.models.gyroswin import GyroSwinMultitask
    from neugk.models.layers import ContinuousConditionEmbed

    ckpt_dim = sd["df_unet.down_blocks.0.swin_att.blocks.0.attn.proj.bias"].shape[0]
    ckpt_in_channels = sd["df_unet.vel_pe.pos_embed"].shape[-1]
    ckpt_real_potens = sd["phi_unet.unpatch.expansion.bias"].shape[0] == 1

    df_base_resolution = list(runner.trainset.resolution)
    phi_base_resolution = list(runner.trainset.phi_resolution)
    n_cond = len(gs_cfg.conditioning)

    cond_fn = ContinuousConditionEmbed(128, n_cond)
    flux_cond_fn = ContinuousConditionEmbed(128, n_cond) if gs_cfg.swin.flux_conditioning else None
    outputs = [k for k in gs_cfg.loss_weights if gs_cfg.loss_weights[k] > 0.0 or gs_cfg.loss_scheduler[k]]

    print(f"Checkpoint: dim={ckpt_dim}, in_channels={ckpt_in_channels}, real_potens={ckpt_real_potens}")

    gyroswin_model = GyroSwinMultitask(
        dim=ckpt_dim,
        outputs=outputs,
        df_base_resolution=df_base_resolution,
        phi_base_resolution=phi_base_resolution,
        df_patch_size=gs_cfg.swin.patch_size,
        phi_patch_size=gs_cfg.swin.phi_patch_size,
        df_window_size=gs_cfg.swin.window_size,
        phi_window_size=gs_cfg.swin.phi_window_size,
        depth=gs_cfg.swin.depth,
        num_heads=gs_cfg.swin.num_heads,
        in_channels=ckpt_in_channels,
        out_channels=ckpt_in_channels,
        num_layers=gs_cfg.num_layers,
        use_checkpoint=gs_cfg.swin.gradient_checkpoint,
        drop_path=gs_cfg.swin.drop_path,
        use_abs_pe=gs_cfg.swin.use_abs_pe,
        c_multiplier=gs_cfg.swin.c_multiplier,
        modulation=gs_cfg.swin.modulation,
        merging_hidden_ratio=gs_cfg.swin.merging_hidden_ratio,
        unmerging_hidden_ratio=gs_cfg.swin.unmerging_hidden_ratio,
        act_fn=getattr(torch.nn, gs_cfg.swin.act_fn),
        patch_skip=gs_cfg.swin.patch_skip,
        decouple_mu=gs_cfg.decouple_mu,
        swin_bottleneck=gs_cfg.swin.swin_bottleneck,
        use_rpb=gs_cfg.swin.use_rpb,
        use_rope=gs_cfg.swin.use_rope,
        latent_cross_attn=gs_cfg.swin.latent_cross_attn,
        real_potens=ckpt_real_potens,
        flux_reduce=gs_cfg.swin.flux_reduce,
        flux_num_heads=gs_cfg.swin.flux_num_heads,
        flux_depth=gs_cfg.swin.flux_depth,
        cond_embed=cond_fn,
        flux_cond_embed=flux_cond_fn,
        conditioning=list(gs_cfg.conditioning),
        init_weights=gs_cfg.swin.init_weights,
        patching_init_weights=gs_cfg.swin.patching_init_weights,
        cond_init_weights=gs_cfg.swin.cond_init_weights,
    )
    info = gyroswin_model.load_state_dict(sd, strict=False)
    if info.unexpected_keys:
        print(f"Unexpected: {info.unexpected_keys[:5]}...")
    if info.missing_keys:
        print(f"Missing: {info.missing_keys[:5]}...")
    gyroswin_model.eval().to(runner.device)
    print(f"GyroSwin: {sum(p.numel() for p in gyroswin_model.parameters())/1e6:.1f}M")
    feature_fn = lambda b, device, **kw: extract_gyroswin_latents(gyroswin_model, b, device, **kw)
    fid_label = "GyroSwin-FID"
else:
    feature_fn = None
    fid_label = "AE-FID"
print(f"Mode: {fid_label}")

In [ ]:
valset = runner.valsets[0]
t_start, t_step, n_snap = 100, 20, 5

_cond_meta_map_fid = {"itg": "ion_temp_grad", "dg": "density_grad"}
_cond_keys_fid = sorted(runner.cfg.model.conditioning)
flat_idx_map = {(f, t): idx for idx, (f, t) in valset.flat_index_to_file_and_tstep.items()}

import re as _re

all_dfs, all_conds, all_traj_idx = [], [], []
traj_labels = []

for fi in range(len(valset.files)):
    meta = valset.metadata[fi]
    fpath = valset.files[fi]
    m = _re.search(r"iteration_(\d+)", fpath)
    label = f"iter_{m.group(1)}" if m else f"f{fi}"

    cond_vals = [float(np.squeeze(meta[_cond_meta_map_fid.get(k, k)])) for k in _cond_keys_fid]

    snap_indices = [t_start + j * t_step for j in range(n_snap)]
    n_ts = len(meta["timesteps"])
    snap_indices = [s for s in snap_indices if s < n_ts]

    count = 0
    for si in snap_indices:
        t_idx = si - valset.offsets[fi]
        if t_idx < 0 or (fi, t_idx) not in flat_idx_map:
            continue
        sample = valset[flat_idx_map[(fi, t_idx)]]
        if sample.df is None:
            continue
        all_dfs.append(sample.df)
        all_conds.append(torch.tensor(cond_vals + [float(meta["timesteps"][si])], dtype=torch.float32))
        all_traj_idx.append(len(traj_labels))
        count += 1

    if count >= 4:
        traj_labels.append(label)
        print(f"  {label}: {count} snapshots")
    else:
        for _ in range(count):
            all_dfs.pop()
            all_conds.pop()
            all_traj_idx.pop()

all_feats = []
for i in tqdm(range(0, len(all_dfs), GEN_BATCH_SIZE), desc="extracting gyroswin features"):
    batch_df = torch.stack(all_dfs[i:i+GEN_BATCH_SIZE])
    batch_cond = torch.stack(all_conds[i:i+GEN_BATCH_SIZE])
    all_feats.append(extract_gyroswin_latents(gyroswin_model, batch_df, device=runner.device, condition=batch_cond))
all_feats = np.concatenate(all_feats)
all_traj_idx = np.array(all_traj_idx)

traj_feats = {}
for ti, label in enumerate(traj_labels):
    mask = all_traj_idx == ti
    traj_feats[label] = all_feats[mask]

n_traj = len(traj_labels)
fid_matrix = np.full((n_traj, n_traj), np.nan)

for i, li in enumerate(traj_labels):
    for j, lj in enumerate(traj_labels):
        fi, fj = traj_feats[li], traj_feats[lj]
        n_min = min(len(fi), len(fj))
        if n_min < 4:
            continue
        mu_i, sig_i = compute_statistics(fi[:n_min])
        mu_j, sig_j = compute_statistics(fj[:n_min])
        fid_matrix[i, j] = compute_fid(mu_i, sig_i, mu_j, sig_j)

fig, ax = plt.subplots(figsize=(max(6, n_traj * 0.8), max(5, n_traj * 0.7)))
im = ax.imshow(fid_matrix, cmap="viridis")
ax.set_xticks(range(n_traj))
ax.set_xticklabels(traj_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(n_traj))
ax.set_yticklabels(traj_labels, fontsize=8)

for i in range(n_traj):
    for j in range(n_traj):
        if np.isfinite(fid_matrix[i, j]):
            ax.text(j, i, f"{fid_matrix[i,j]:.1f}", ha="center", va="center", fontsize=7,
                    color="white" if fid_matrix[i,j] > np.nanmedian(fid_matrix) else "black")

plt.colorbar(im, ax=ax, label="FID")
ax.set(title="pairwise GyroSwin-FID across ID validation trajectories")
fig.tight_layout()

In [ ]:
runner.model.eval()

if FID_MODE == "ae":
    real_lats, real_conds = [], []
    for (fi, ti), s in tqdm(runner.trainset.precomputed_latents.items(), desc="Real latents"):
        real_lats.append(np.array(s["x"]).reshape(-1))
        real_conds.append(np.array([s["itg"], s["dg"], s["s_hat"], s["q"]]))
        if FID_MAX_SAMPLES and len(real_lats) >= FID_MAX_SAMPLES: break
    X_real, C_real = np.stack(real_lats), np.stack(real_conds)
    idx = np.random.choice(len(X_real), len(X_real), replace=True)
    gen_lats = []
    for i in tqdm(range(0, len(X_real), GEN_BATCH_SIZE), desc="Gen latents"):
        c = torch.tensor(C_real[idx[i:i+GEN_BATCH_SIZE]], dtype=torch.float32, device=runner.device)
        with torch.no_grad():
            z = runner.sample(c, latent_only=True, steps=N_DENOISING_STEPS)
            gen_lats.append(z.cpu().numpy().reshape(z.shape[0], -1))
    X_gen = np.concatenate(gen_lats)

elif FID_MODE == "gyroswin":
    n = FID_MAX_SAMPLES or min(512, len(runner.valsets[0]))
    real_s = [runner.valsets[0][i].df for i in tqdm(range(n), desc="Loading real")]
    gen_s, gen_conds_list = [], []
    _cond_meta_map = {"itg": "ion_temp_grad", "dg": "density_grad"}
    _cond_keys = sorted(runner.cfg.model.conditioning)
    real_conds_list = []
    for idx in range(n):
        fi, _ = runner.valsets[0].flat_index_to_file_and_tstep[idx]
        meta = runner.valsets[0].metadata[fi]
        real_conds_list.append(torch.tensor(
            [float(np.squeeze(meta[_cond_meta_map.get(k, k)])) for k in _cond_keys],
            dtype=torch.float32))
    conds = torch.stack(real_conds_list)
    for i in tqdm(range(0, n, GEN_BATCH_SIZE), desc="Generating"):
        c = conds[i:i+GEN_BATCH_SIZE].to(runner.device)
        with torch.no_grad():
            p = runner.sample(c, steps=N_DENOISING_STEPS, latent_only=False)
            for j in range(p["df"].shape[0]):
                gen_s.append(p["df"][j].cpu())
                gen_conds_list.append(conds[i+j])
    print(f"Extracting features...")
    _, X_real, X_gen = compute_fid_on_latents(
        real_s, gen_s, feature_fn, device=runner.device, n_components=FID_N_COMPONENTS,
        real_conditions=real_conds_list, gen_conditions=gen_conds_list)

print(f"Real: {X_real.shape}, Gen: {X_gen.shape}")

In [ ]:
X_rf, X_gf = X_real, X_gen
if FID_N_COMPONENTS and FID_N_COMPONENTS < X_real.shape[1]:
    pca = PCA(n_components=FID_N_COMPONENTS)
    X_rf = pca.fit_transform(X_real)
    X_gf = pca.transform(X_gen)
    print(f"PCA: {X_real.shape[1]}->{FID_N_COMPONENTS} ({pca.explained_variance_ratio_.sum():.1%} var)")

mu_r, sig_r = compute_statistics(X_rf)
mu_g, sig_g = compute_statistics(X_gf)
fid_global = compute_fid(mu_r, sig_r, mu_g, sig_g)
print(f"Global {fid_label}: {fid_global:.4f}")

In [ ]:
per_traj_fids = {}
if FID_MODE == "ae":
    ptr = defaultdict(list)
    for (fi, ti), s in runner.trainset.precomputed_latents.items():
        ptr[fi].append(np.array(s["x"]).reshape(-1))
    for fi, rl in tqdm(ptr.items(), desc="Per-traj FID"):
        Xt = np.stack(rl)
        if len(Xt) < 10: continue
        sk = next(k for k in runner.trainset.precomputed_latents if k[0] == fi)
        s = runner.trainset.precomputed_latents[sk]
        c = torch.tensor([[s["itg"], s["dg"], s["s_hat"], s["q"]]],
                          dtype=torch.float32, device=runner.device).expand(len(Xt), -1)
        with torch.no_grad():
            zg = runner.sample(c, latent_only=True, steps=N_DENOISING_STEPS)
            Xg = zg.cpu().numpy().reshape(zg.shape[0], -1)
        if FID_N_COMPONENTS and FID_N_COMPONENTS < Xt.shape[1]:
            Xt, Xg = pca.transform(Xt), pca.transform(Xg)
        m1, s1 = compute_statistics(Xt)
        m2, s2 = compute_statistics(Xg)
        per_traj_fids[fi] = compute_fid(m1, s1, m2, s2)
    print(f"Per-traj mean: {np.mean(list(per_traj_fids.values())):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
mr, mg = X_real.mean(0), X_gen.mean(0)
axes[0].scatter(mr, mg, alpha=0.3, s=10, color="#264653")
lim = [min(mr.min(), mg.min()), max(mr.max(), mg.max())]
axes[0].plot(lim, lim, "r--", alpha=0.8)
axes[0].set(title="Feature means", xlabel="Real", ylabel="Gen")
sr, sg = X_real.std(0), X_gen.std(0)
axes[1].scatter(sr, sg, alpha=0.3, s=10, color="#e9c46a")
lim = [min(sr.min(), sg.min()), max(sr.max(), sg.max())]
axes[1].plot(lim, lim, "r--", alpha=0.8)
axes[1].set(title="Feature stds", xlabel="Real", ylabel="Gen")
if per_traj_fids:
    axes[2].hist(list(per_traj_fids.values()), bins=30, color="#2a9d8f", edgecolor="k", alpha=0.8)
    axes[2].axvline(fid_global, color="red", ls="--", lw=1.5, label=f"Global={fid_global:.1f}")
    axes[2].set(title=f"Per-traj {fid_label}", xlabel="FID")
    axes[2].legend()
else:
    axes[2].text(0.5, 0.5, "N/A", ha="center", va="center", transform=axes[2].transAxes)
fig.suptitle(f"{fid_label} = {fid_global:.2f}", fontweight="bold")
fig.tight_layout()

## 3. Warm Restart Evaluation

In [ ]:
from gyaradax import gk_from_gkw_dir

In [ ]:
N_EVAL_STEPS = 1000
WARM_RESTART_ITERATIONS = [13]

warm_results = {}
_cond_keys = sorted(runner.cfg.model.conditioning)

for iteration in WARM_RESTART_ITERATIONS:
    print(f"\n{'=' * 88}\nIteration {iteration}\n{'=' * 88}")
    df_gt, geometry, params, state_init, pre = gk_from_gkw_dir(
        os.path.join(GKW_RAW_DIR, f"iteration_{iteration}"), mixed_precision=True, k_index=80
    )

    _param_map = {"itg": "rlt", "dg": "rln", "s_hat": "shat", "q": "q"}
    cond = torch.tensor(
        [[float(getattr(params, _param_map[k])) for k in _cond_keys]],
        dtype=torch.float32, device=runner.device,
    )

    matching = [f for f in runner.trainset.files if f"iteration_{iteration}" in f]
    fi = runner.trainset.files.index(matching[0]) if matching else -1

    with torch.no_grad():
        DF_PRED = runner.sample(cond, latent_only=False, steps=N_DENOISING_STEPS)["df"][0].cpu().numpy()

    df_warm = from_model_space(DF_PRED)
    log_gt, log_warm = run_trajectory_pair(
        df_gt, df_warm, geometry, params, pre, state_init,
        N_EVAL_STEPS, f"iter-{iteration}", chunk_size=10,
        backend="jax", print_every=100,
    )
    ref_mean, ref_std = load_reference_flux(GKW_RAW_DIR, iteration)
    print(f"  Reference flux: mean={ref_mean:.3e}, std={ref_std:.3e}")
    ttc = time_to_convergence(log_warm, log_gt, log_gt["time"],
                              ref_flux_mean=ref_mean, ref_flux_std=ref_std)
    ttc_flux, ttc_spec = ttc["flux"], ttc["ky_spec"]
    fid_val = per_traj_fids.get(fi, fid_global if 'fid_global' in dir() else np.nan)

    warm_results[iteration] = dict(
        fid=fid_val, ttc_flux=ttc_flux, ttc_spec=ttc_spec, log_gt=log_gt, log_warm=log_warm, df_pred=DF_PRED,
    )
    print(f"  TTC_flux={ttc_flux:.3f}, TTC_spec={ttc_spec:.3f}, FID={fid_val:.4f}")

In [ ]:
from neugk.plot_utils import plot_nd

for it, res in warm_results.items():
    df_ic = torch.tensor(res["df_pred"], dtype=torch.float32)

    df_final = torch.tensor(to_model_space(res["log_warm"]["df_final"]), dtype=torch.float32)

    print(f"\nIteration {it}: IC vs final warm-start state")
    fig = plot_nd(df_ic, df_final, to_wandb=False)
    fig.suptitle(f"Iter {it}: Generated IC (left) vs Warm-start final (right)",
                 fontsize=11, y=1.01)

In [ ]:
n = len(warm_results)
fig, axes = plt.subplots(n, 3, figsize=(15, 4*n), squeeze=False)
snap_colors = ["#2a9d8f", "#e76f51", "#264653"]

for row, (it, res) in enumerate(warm_results.items()):
    lg, lw, t = res["log_gt"], res["log_warm"], res["log_gt"]["time"]

    n_t = len(t)
    snap_idx = [min(5, n_t-1), n_t//2, n_t-1]
    for j, si in enumerate(snap_idx):
        ky_gt = np.log10(np.maximum(lg["ky_spec"][si], 1e-30))
        ky_w  = np.log10(np.maximum(lw["ky_spec"][si], 1e-30))
        lbl = f"t={float(t[si]):.1f}"
        axes[row,0].plot(ky_gt, "-",  color=snap_colors[j], lw=1.2, alpha=0.7, label=f"GT {lbl}")
        axes[row,0].plot(ky_w,  "--", color=snap_colors[j], lw=1.2, alpha=0.9, label=f"warm {lbl}")
    axes[row,0].set(title=f"iter {it}: $k_y$ spectrum", xlabel="$k_y$ mode", ylabel="log$_{10}$(E)")
    axes[row,0].legend(fontsize=6, ncol=2); axes[row,0].grid(True, alpha=0.15)

    ref_mean, ref_std = load_reference_flux(GKW_RAW_DIR, it)
    axes[row,1].plot(t, lg["eflux"], "k", lw=0.8, alpha=0.5, label="GT")
    axes[row,1].plot(t, lw["eflux"], lw=1, label="warm")
    axes[row,1].axhspan(ref_mean - ref_std, ref_mean + ref_std, color="k", alpha=0.08, label=f"ref ±1σ")
    axes[row,1].axhline(ref_mean, color="k", ls=":", lw=0.8)
    if res["ttc_flux"] < np.inf:
        axes[row,1].axvline(t[0]+res["ttc_flux"], color="green", ls="-", lw=1.5, label=f"TTC_flux={res['ttc_flux']:.1f}")
    if res["ttc_spec"] < np.inf:
        axes[row,1].axvline(t[0]+res["ttc_spec"], color="blue", ls="--", lw=1.5, label=f"TTC_spec={res['ttc_spec']:.1f}")
    axes[row,1].set_title(f"iter {it}: flux"); axes[row,1].legend(fontsize=6); axes[row,1].grid(True, alpha=0.15)

    n_compare = min(len(lw["ky_spec"]), len(lg["ky_spec"]))
    r_ts = [pearsonr(np.log10(np.maximum(lw["ky_spec"][i],1e-30)),
                      np.log10(np.maximum(lg["ky_spec"][i],1e-30)))[0]
            for i in range(n_compare)]
    axes[row,2].plot(t[:n_compare], r_ts, lw=1)
    axes[row,2].axhline(0.95, color="gray", ls="--", lw=0.8, label="0.95")
    if res["ttc_spec"] < np.inf:
        axes[row,2].axvline(t[0]+res["ttc_spec"], color="blue", ls="--", lw=1, alpha=0.5)
    axes[row,2].set_title(f"iter {it}: Pearson($k_y$)"); axes[row,2].set_ylim(-0.1,1.05)
    axes[row,2].legend(fontsize=7); axes[row,2].grid(True, alpha=0.15)

for ax in axes[-1]: ax.set_xlabel(r"time $[v_{th}/R]$")
fig.tight_layout()

## 4. FID vs. Time-to-Convergence

In [ ]:
fv, tv, lb = [], [], []
for it, res in warm_results.items():
    if np.isfinite(res["fid"]) and np.isfinite(res["ttc_flux"]):
        fv.append(res["fid"]); tv.append(res["ttc_flux"]); lb.append(f"iter_{it}")
fv, tv = np.array(fv), np.array(tv)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(fv, tv, s=80, zorder=3, edgecolors="k", linewidth=0.5)
for i, l in enumerate(lb): ax.annotate(l, (fv[i], tv[i]), textcoords="offset points", xytext=(8,4), fontsize=9)
if len(fv)>=3:
    co = np.polyfit(fv, tv, 1)
    xf = np.linspace(fv.min(), fv.max(), 100)
    ax.plot(xf, np.polyval(co, xf), "r--", lw=1.5)
    r, p = pearsonr(fv, tv)
    ax.text(0.05, 0.95, f"r={r:.3f}, p={p:.3f}", transform=ax.transAxes, fontsize=12, va="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
ax.set(xlabel=fid_label, ylabel=r"TTC $[v_{th}/R]$", title=f"{fid_label} vs TTC")
ax.grid(True, alpha=0.15)
fig.tight_layout()

In [ ]:
print(f"{'Iter':>8} {'FID':>10} {'TTC_flux':>10} {'TTC_spec':>10} {'RelErr':>10}")
print("-"*54)
for it, r in warm_results.items():
    f = f"{r['fid']:.4f}" if np.isfinite(r['fid']) else 'N/A'
    tf = f"{r['ttc_flux']:.3f}" if np.isfinite(r['ttc_flux']) else 'inf'
    ts = f"{r['ttc_spec']:.3f}" if np.isfinite(r['ttc_spec']) else 'inf'

## 5. Extended Correlation Analysis

In [ ]:
extended_metrics = {}
for it, res in warm_results.items():
    m = dict(fid=res["fid"], ttc=res["ttc_flux"])
    lg, lw = res["log_gt"], res["log_warm"]
    gm = np.mean(lg["eflux"][-80:])
    m["integral_flux_err"] = float(abs(lw["eflux"][0] - gm) / max(abs(gm), 1e-30))
    ky_lg = np.log10(np.maximum(np.mean(lg["ky_spec"][-80:], 0), 1e-30))
    ky_lw = np.log10(np.maximum(lw["ky_spec"][0], 1e-30))
    r_ky, _ = pearsonr(ky_lw, ky_lg) if len(ky_lg)>1 else (0.,1.)
    m["kyspec_pearson_init"] = float(r_ky)
    m["kyspec_rmse_init"] = float(np.sqrt(np.mean((ky_lw - ky_lg)**2)))
    kx_lg = np.log10(np.maximum(np.mean(lg["kx_spec"][-80:], 0), 1e-30))
    kx_lw = np.log10(np.maximum(lw["kx_spec"][0], 1e-30))
    m["kxspec_rmse_init"] = float(np.sqrt(np.mean((kx_lw - kx_lg)**2)))
    m["probe_flux_rmse"] = np.nan
    extended_metrics[it] = m
display(pd.DataFrame(extended_metrics).T.round(4))

In [ ]:
_ = plot_correlation_grid(extended_metrics)